# Transaction Fraud Intelligence — first modelling experiment

**Status:** synthetic research prototype. Running this notebook produces the results; no performance is assumed in advance.

The experiment asks: **does customer history improve fraud ranking within a limited daily review budget?**

We generate normal customers and legitimate changes, inject three fraud mechanisms, build features using only earlier observations, and compare rules, logistic regression, transaction-only CatBoost, and CatBoost with history.

**How to run:** choose a CPU runtime in Google Colab and select **Runtime → Run all**. The notebook is self-contained; you do not need to clone the repository or provide a dataset. Run the dependency cell before importing scientific packages in a fresh runtime.

Amounts are fictional **INR**. Fraud rates, customer behaviour, outcome latency and label delays are engineering assumptions. Results do not establish real-world accuracy or money saved. The final chronological test period stays unevaluated while we develop the model.

The last cell shows a ZIP download link. Download it before closing the runtime. Keep this notebook executable; readable specifications live in Markdown.

In [ ]:
import os
import sys
import subprocess

DEPENDENCIES = [
    "numpy==2.2.6", "pandas==2.2.3", "scipy==1.15.3",
    "scikit-learn==1.6.1", "catboost==1.2.8",
    "matplotlib==3.10.1", "joblib==1.4.2",
]
# CI installs the same packages before starting the notebook kernel.
if os.environ.get("FRAUD_SKIP_INSTALL") != "1":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *DEPENDENCIES])
print("Dependencies ready. Use a fresh runtime if these packages were already imported.")

## 1. Configuration

The default is 1,200 customers across 120 days. A smaller automated smoke run uses 360 customers and fewer boosting iterations, preserving the same timeline and validation logic.

These are simulation settings, not estimates of a real payment portfolio. Change the seed only as a declared new experiment; do not search seeds for flattering results.

In [ ]:
from pathlib import Path
from collections import defaultdict, deque
from itertools import groupby
import heapq
import hashlib
import importlib.metadata
import json
import math
import shutil
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from IPython.display import display, FileLink
from catboost import CatBoostClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve

SMOKE = os.environ.get("FRAUD_SMOKE_TEST") == "1"
CFG = {
    "seed": 42,
    "customers": 360 if SMOKE else 1200,
    "days": 120,
    "merchants": 240,
    "fraud_customer_fraction": 0.35,
    "label_delay_days": 7,
    "review_budgets": [20, 50, 100],
    "main_review_budget": 50,
    "boosting_iterations": 100 if SMOKE else 600,
    "mode": "smoke" if SMOKE else "development",
}
START = pd.Timestamp("2025-01-01", tz="UTC")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
OUT = Path("outputs") / ("fraud_v1_" + RUN_ID)
OUT.mkdir(parents=True, exist_ok=False)
pd.set_option("display.max_columns", 20)
print(json.dumps(CFG, indent=2))
print("Run directory:", OUT.resolve())

## 2. Simulate customers, merchants and payment attempts

Customers have persistent spend distributions, preferred hours, favourite merchants, and devices. Normal activity includes travel, permanent phone changes, large purchases, and legitimate bursts.

Fraud mechanisms:
- **Account takeover:** several purchases, sometimes with an existing device.
- **Card testing:** a rapid sequence of smaller attempts, with variable failures.
- **Low-and-slow:** several ordinary-looking purchases spread across days.

Legitimate bursts and large purchases overlap with fraudulent patterns. This overlap reduces trivial separation but does not eliminate simulator bias.

Hidden customer preferences and scenario labels are stored for auditing only. Current outcomes become available after a simulated delay; confirmed labels become available seven days later. All attempted amounts, including failed attempts, contribute to attempt-history features.

In [ ]:
CATEGORIES = ["groceries", "retail", "dining", "travel", "digital", "utilities"]
COUNTRIES = ["IN", "SG", "AE", "GB", "US"]

def simulate(cfg):
    rng = np.random.default_rng(cfg["seed"])
    merchant_ids = np.array(["M%04d" % i for i in range(cfg["merchants"])])
    merchants = pd.DataFrame({
        "merchant_id": merchant_ids,
        "category": [CATEGORIES[i % len(CATEGORIES)] for i in range(len(merchant_ids))],
    })
    merchant_category = dict(zip(merchants.merchant_id, merchants.category))
    customers, events, profiles = [], [], []
    horizon = cfg["days"] * 86400

    def add(cid, sec, amount, device, country, merchant, fraud, scenario, context, fail_p):
        sec = int(sec)
        if not 0 <= sec < horizon:
            return
        events.append({
            "customer_id": cid, "event_second": sec,
            "amount": round(float(np.clip(amount, 1, 200000)), 2),
            "device_id": str(device), "country": str(country),
            "merchant_id": str(merchant), "category": merchant_category[str(merchant)],
            "status": "failed" if rng.random() < fail_p else "succeeded",
            "outcome_delay_seconds": int(rng.integers(2, 121)),
            "is_fraud": int(fraud), "fraud_scenario": scenario,
            "legitimate_context": context,
        })

    for i in range(cfg["customers"]):
        cid = "C%05d" % i
        base = float(np.clip(rng.lognormal(np.log(850), 0.8), 120, 7000))
        spread = float(rng.uniform(0.3, 0.85))
        rate = float(np.clip(rng.gamma(2.0, 0.25), 0.12, 1.3))
        hour = int(rng.integers(7, 24))
        home = str(rng.choice(COUNTRIES, p=[0.80, 0.06, 0.06, 0.04, 0.04]))
        favourites = rng.choice(merchant_ids, size=6, replace=False)
        device0, device1 = "phone_" + cid, "replacement_" + cid
        phone_day = int(rng.integers(25, cfg["days"] - 10)) if rng.random() < 0.30 else cfg["days"] + 1
        travel_day = int(rng.integers(15, cfg["days"] - 8)) if rng.random() < 0.25 else cfg["days"] + 1
        travel_country = str(rng.choice([x for x in COUNTRIES if x != home]))
        profile = dict(customer_id=cid, base=base, spread=spread, rate=rate,
                       hour=hour, home=home, favourites=favourites,
                       device0=device0, device1=device1, phone_day=phone_day)
        profiles.append(profile)
        customers.append({
            "customer_id": cid, "simulator_base_amount": base,
            "simulator_spread": spread, "simulator_daily_rate": rate,
            "simulator_preferred_hour": hour, "simulator_home_country": home,
        })
        n = int(rng.poisson(rate * cfg["days"]))
        days = rng.integers(0, cfg["days"], size=n)
        hours = np.mod(rng.normal(hour, 3.0, size=n), 24)
        for day, event_hour in zip(days, hours):
            context = "routine"
            country = home
            device = device1 if day >= phone_day else device0
            if phone_day <= day < phone_day + 5:
                context = "new_phone"
            if travel_day <= day < travel_day + 6:
                country, context = travel_country, "travel"
            if rng.random() < 0.025:
                device = "household_%04d" % (i // 4)
            amount = base * rng.lognormal(0, spread)
            if rng.random() < 0.018:
                amount *= rng.uniform(2, 7)
                context = "large_purchase"
            merchant = rng.choice(favourites if rng.random() < 0.85 else merchant_ids)
            add(cid, day * 86400 + int(event_hour * 3600), amount, device,
                country, merchant, 0, "legitimate", context, 0.05)

        # Legitimate bursts overlap with testing/velocity signals.
        if rng.random() < 0.35:
            day = int(rng.integers(10, cfg["days"]))
            sec = day * 86400 + hour * 3600
            for j in range(int(rng.integers(4, 11))):
                add(cid, sec + j * 70, base * rng.uniform(0.08, 2.2),
                    device1 if day >= phone_day else device0, home,
                    rng.choice(favourites), 0, "legitimate", "legitimate_burst", 0.25)

    # Assignment is independent of model inputs; raw customer/device IDs are excluded.
    selected = rng.choice(len(profiles),
                          size=max(3, int(len(profiles) * cfg["fraud_customer_fraction"])),
                          replace=False)
    scenarios = ["account_takeover", "card_testing", "low_and_slow"]
    for k, index in enumerate(selected):
        p = profiles[int(index)]
        scenario = scenarios[k % 3]
        day = int(rng.integers(12, cfg["days"]))
        current_device = p["device1"] if day >= p["phone_day"] else p["device0"]
        device = current_device if rng.random() < 0.45 else "shared_actor_%03d" % int(rng.integers(0, 30))
        country = p["home"] if rng.random() < 0.75 else str(rng.choice(COUNTRIES))
        base_sec = day * 86400 + int(rng.uniform(0, 24) * 3600)
        if scenario == "account_takeover":
            count, step, fail_p = int(rng.integers(2, 7)), int(rng.integers(120, 1800)), 0.18
        elif scenario == "card_testing":
            count, step, fail_p = int(rng.integers(5, 13)), int(rng.integers(15, 150)), 0.45
        else:
            count, step, fail_p = int(rng.integers(3, 9)), int(rng.integers(1, 4)) * 86400, 0.06
        for j in range(count):
            if scenario == "account_takeover":
                amount = p["base"] * rng.uniform(1.2, 6.0)
            elif scenario == "card_testing":
                amount = p["base"] * rng.uniform(0.02, 0.35)
            else:
                amount = p["base"] * rng.lognormal(0, p["spread"] * 0.7)
            merchant = rng.choice(p["favourites"] if rng.random() < 0.35 else merchant_ids)
            add(p["customer_id"], base_sec + j * step, amount, device, country,
                merchant, 1, scenario, "not_applicable", fail_p)

    tx = pd.DataFrame(events)
    tx["timestamp"] = START + pd.to_timedelta(tx.pop("event_second"), unit="s")
    tx["outcome_available_at"] = tx.timestamp + pd.to_timedelta(tx.pop("outcome_delay_seconds"), unit="s")
    tx["label_available_at"] = tx.timestamp + pd.Timedelta(days=cfg["label_delay_days"])
    tx = tx.sort_values("timestamp", kind="stable").reset_index(drop=True)
    tx.insert(0, "transaction_id", ["T%08d" % i for i in range(len(tx))])
    return pd.DataFrame(customers), merchants, tx

customers, merchants, transactions = simulate(CFG)
assert transactions.transaction_id.is_unique
assert transactions.timestamp.is_monotonic_increasing
assert (transactions.amount > 0).all()
assert (transactions.outcome_available_at > transactions.timestamp).all()
assert (transactions.label_available_at > transactions.timestamp).all()
assert set(transactions.customer_id) <= set(customers.customer_id)
assert set(transactions.merchant_id) <= set(merchants.merchant_id)
print("Synthetic events generated:", len(transactions))
print("Scenario labels and hidden profiles will not be passed to either model.")

## 3. Build features as transactions arrive

For a decision at time t, history contains only events strictly before t. Transactions with identical timestamps are scored as a batch before any of them updates history. Failed outcomes enter the available-information queue only when their availability timestamp is reached.

The 30-day baseline uses prior attempted amounts. With fewer than three observations, personal amount comparisons remain missing; the model receives history-coverage features. All transformations learned by logistic regression are fitted on training rows.

Raw IDs support joins and history calculations but do not enter a model. Simulator parameters, fraud scenarios, legitimate-context tags, current status and labels are excluded from the feature matrix.

In [ ]:
OBSERVABLE_COLUMNS = [
    "transaction_id", "timestamp", "customer_id", "merchant_id",
    "device_id", "amount", "country", "category", "status", "outcome_available_at",
]
NUMERIC_FEATURES = [
    "log_amount", "hour_sin", "hour_cos", "prior_count", "prior_count_30d",
    "attempts_1h", "attempts_24h", "log_attempt_value_24h",
    "log_prior_median_30d", "amount_ratio_30d", "amount_z_30d",
    "history_days", "hours_since_previous", "new_device", "new_country",
    "new_merchant", "known_failures_1h", "prior_accounts_on_device", "hour_deviation",
]
CATEGORICAL_FEATURES = ["category", "country"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
CURRENT_FEATURES = ["log_amount", "hour_sin", "hour_cos", "category", "country"]

def build_features(tx):
    events = tx[OBSERVABLE_COLUMNS].sort_values(["timestamp", "transaction_id"])
    state, device_accounts, pending_outcomes, records = {}, defaultdict(set), [], []
    known_failures = defaultdict(deque)
    ns_hour = 3600 * 10**9
    ns_day = 24 * ns_hour
    for timestamp, batch_iter in groupby(events.itertuples(index=False), key=lambda r: r.timestamp):
        batch = list(batch_iter)
        t = int(timestamp.value)
        while pending_outcomes and pending_outcomes[0][0] <= t:
            available, event_id, cid = heapq.heappop(pending_outcomes)
            known_failures[cid].append(available)
        for r in batch:
            cid = r.customer_id
            if cid not in state:
                state[cid] = {
                    "history": deque(), "count": 0, "devices": set(),
                    "countries": set(), "merchants": set(),
                    "first": t, "last": None, "sin_sum": 0.0, "cos_sum": 0.0,
                }
            s = state[cid]
            h = s["history"]
            while h and h[0][0] < t - 30 * ns_day:
                h.popleft()
            failures = known_failures[cid]
            while failures and failures[0] < t - ns_hour:
                failures.popleft()
            amounts = np.array([a for _, a in h], dtype=float)
            n30 = len(amounts)
            median = float(np.median(amounts)) if n30 else np.nan
            enough = n30 >= 3
            last24 = [a for old_t, a in h if old_t >= t - ns_day]
            angle = 2 * np.pi * (timestamp.hour + timestamp.minute / 60) / 24
            sine, cosine = float(np.sin(angle)), float(np.cos(angle))
            norm = math.hypot(s["sin_sum"], s["cos_sum"])
            hour_dev = np.nan
            if s["count"] >= 3 and norm > 1e-8:
                dot = (sine * s["sin_sum"] + cosine * s["cos_sum"]) / norm
                hour_dev = float(np.arccos(np.clip(dot, -1, 1)) * 12 / np.pi)
            records.append({
                "transaction_id": r.transaction_id,
                "log_amount": float(np.log1p(r.amount)),
                "hour_sin": sine, "hour_cos": cosine,
                "prior_count": s["count"], "prior_count_30d": n30,
                "attempts_1h": sum(old_t >= t - ns_hour for old_t, _ in h),
                "attempts_24h": len(last24),
                "log_attempt_value_24h": float(np.log1p(sum(last24))),
                "log_prior_median_30d": float(np.log1p(median)) if n30 else np.nan,
                "amount_ratio_30d": float(r.amount / median) if enough else np.nan,
                "amount_z_30d": float((r.amount - amounts.mean()) / (amounts.std() + 50)) if enough else np.nan,
                "history_days": (t - s["first"]) / ns_day,
                "hours_since_previous": (t - s["last"]) / ns_hour if s["last"] is not None else np.nan,
                "new_device": int(r.device_id not in s["devices"]),
                "new_country": int(r.country not in s["countries"]),
                "new_merchant": int(r.merchant_id not in s["merchants"]),
                "known_failures_1h": len(failures),
                "prior_accounts_on_device": len(device_accounts[r.device_id]),
                "hour_deviation": hour_dev,
                "category": str(r.category), "country": str(r.country),
            })
        # Update only after every event at this timestamp has been scored.
        for r in batch:
            s = state[r.customer_id]
            angle = 2 * np.pi * (timestamp.hour + timestamp.minute / 60) / 24
            s["history"].append((t, float(r.amount)))
            s["count"] += 1
            s["last"] = t
            s["devices"].add(r.device_id)
            s["countries"].add(r.country)
            s["merchants"].add(r.merchant_id)
            s["sin_sum"] += float(np.sin(angle))
            s["cos_sum"] += float(np.cos(angle))
            device_accounts[r.device_id].add(r.customer_id)
            if r.status == "failed":
                heapq.heappush(pending_outcomes, (
                    int(r.outcome_available_at.value), r.transaction_id, r.customer_id))
    return pd.DataFrame(records).set_index("transaction_id")

features = build_features(transactions)
data = transactions.set_index("transaction_id").join(features, rsuffix="_feature")
X = features[FEATURES].copy()
for col in CATEGORICAL_FEATURES:
    X[col] = X[col].astype(str)
assert X.index.equals(data.index)
assert not set(FEATURES) & {"is_fraud", "fraud_scenario", "legitimate_context",
                           "customer_id", "device_id", "status", "label_available_at"}
assert not np.isinf(X[NUMERIC_FEATURES].to_numpy(dtype=float)).any()
print("Feature matrix:", X.shape)
display(X.head(3))

## 4. Check the timing safeguards

These checks target important failure modes:
1. Appending future events must leave earlier features unchanged.
2. Events at an identical timestamp must not see one another.
3. A previous failed outcome must remain unavailable until its stated release time.
4. Historical amounts must exclude the current transaction.

In [ ]:
def check_timing():
    times = [START, START + pd.Timedelta(seconds=30),
             START + pd.Timedelta(seconds=30), START + pd.Timedelta(seconds=120)]
    probe = pd.DataFrame({
        "transaction_id": ["p0", "p1", "p2", "p3"],
        "timestamp": times, "customer_id": ["c"] * 4,
        "merchant_id": ["m"] * 4, "device_id": ["d"] * 4,
        "amount": [100.0, 200.0, 400.0, 800.0],
        "country": ["IN"] * 4, "category": ["retail"] * 4,
        "status": ["failed", "succeeded", "succeeded", "succeeded"],
        "outcome_available_at": [t + pd.Timedelta(seconds=60) for t in times],
    })
    f = build_features(probe)
    assert f.loc["p1", "prior_count"] == f.loc["p2", "prior_count"] == 1
    assert f.loc["p1", "known_failures_1h"] == 0
    assert f.loc["p3", "known_failures_1h"] == 1
    assert np.isclose(np.expm1(f.loc["p1", "log_prior_median_30d"]), 100)
    assert np.isclose(f.loc["p3", "amount_ratio_30d"], 4.0)
    before = build_features(probe.iloc[:3])
    pd.testing.assert_frame_equal(before, f.loc[before.index])

    boundary = transactions.timestamp.iloc[min(1800, len(transactions) - 1)]
    prefix = transactions.loc[transactions.timestamp <= boundary]
    rebuilt = build_features(prefix)
    pd.testing.assert_frame_equal(rebuilt, features.loc[rebuilt.index])
    return {"future_invariance": "passed", "same_time_batching": "passed",
            "delayed_outcomes": "passed", "current_amount_excluded": "passed"}

timing_checks = check_timing()
print(timing_checks)

## 5. Chronological development periods

Training labels must be available when the model is first fitted. Early-stopping labels must be available before the validation period begins. This creates explicit seven-day label-maturation gaps.

Validation is for development decisions. The final 15% of the timeline is reserved and is not scored or included in performance exports. The lock is a workflow convention, not a security boundary: the simulator still creates those events in memory.

Customer history continues to update using observable events, including unlabelled events in the gaps. No fraud labels are used by the historical feature builder.

In [ ]:
a, b, c = [int(CFG["days"] * fraction) for fraction in (0.60, 0.72, 0.85)]
fit_at = START + pd.Timedelta(days=a)
validation_at = START + pd.Timedelta(days=b)
test_at = START + pd.Timedelta(days=c)
masks = {
    "train": (data.timestamp < fit_at) & (data.label_available_at <= fit_at),
    "early_stop": (data.timestamp >= fit_at) & (data.timestamp < validation_at)
                  & (data.label_available_at <= validation_at),
    "validation": (data.timestamp >= validation_at) & (data.timestamp < test_at),
    "locked_test": data.timestamp >= test_at,
}
assert not (masks["train"] & masks["early_stop"]).any()
assert not (masks["early_stop"] & masks["validation"]).any()
audit = []
for name in ["train", "early_stop", "validation"]:
    part = data.loc[masks[name]]
    if part.is_fraud.nunique() != 2:
        raise ValueError(name + " needs both classes; increase the simulated population.")
    audit.append({
        "period": name, "rows": len(part), "frauds": int(part.is_fraud.sum()),
        "fraud_rate": float(part.is_fraud.mean()),
        "first_event": str(part.timestamp.min()), "last_event": str(part.timestamp.max()),
    })
split_audit = pd.DataFrame(audit)
display(split_audit)
print("Locked test begins:", test_at, "— no predictions or performance computed.")
train_mask, stop_mask, val_mask = [masks[n] for n in ["train", "early_stop", "validation"]]
y = data.is_fraud.astype(int)
split_audit.to_csv(OUT / "split_audit.csv", index=False)

## 6. Train the baselines and the history experiment

Rules are fixed illustrative conditions with a 0–100 point score. Logistic regression uses the full feature set. Two CatBoost models compare current-transaction fields against the full historical feature set.

The comparison is deliberately small: no hyperparameter search, resampling, anomaly ensemble or calibration yet. CatBoost uses the early-stopping period, never the final test period. Scores from the fitted classifiers are uncalibrated model outputs, not verified probabilities.

The history comparison measures the whole added feature group under these training settings. More tuning and repeated temporal experiments would be needed to establish a general advantage.

In [ ]:
RULE_WEIGHTS = {
    "unusual_amount": 25, "new_device_high_amount": 25, "attempt_burst": 20,
    "known_failure_sequence": 15, "shared_device_new_country": 15,
}

def rule_triggers(frame):
    established = frame.prior_count >= 5
    return pd.DataFrame({
        "unusual_amount": established & (frame.amount_ratio_30d >= 4),
        "new_device_high_amount": established & frame.new_device.eq(1) & (frame.amount_ratio_30d >= 2),
        "attempt_burst": frame.attempts_1h >= 4,
        "known_failure_sequence": frame.known_failures_1h >= 2,
        "shared_device_new_country": established & (frame.prior_accounts_on_device >= 3) & frame.new_country.eq(1),
    }, index=frame.index).astype(int)

def rule_scores(frame):
    return rule_triggers(frame).mul(pd.Series(RULE_WEIGHTS)).sum(axis=1).to_numpy(dtype=float)

preprocess = ColumnTransformer([
    ("numeric", Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True)),
        ("scale", StandardScaler()),
    ]), NUMERIC_FEATURES),
    ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=True), CATEGORICAL_FEATURES),
])
logistic = Pipeline([
    ("preprocess", preprocess),
    ("classifier", LogisticRegression(C=1.0, max_iter=1500, solver="lbfgs", random_state=CFG["seed"])),
])
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always", ConvergenceWarning)
    logistic.fit(X.loc[train_mask], y.loc[train_mask])
    if any(issubclass(w.category, ConvergenceWarning) for w in caught):
        raise RuntimeError("Logistic regression did not converge; inspect its numerical features.")

models = {"logistic_history": logistic}
model_columns = {"logistic_history": FEATURES}
training_rows = []
for name, columns in [("catboost_current", CURRENT_FEATURES), ("catboost_history", FEATURES)]:
    model = CatBoostClassifier(
        iterations=CFG["boosting_iterations"], depth=6, learning_rate=0.06,
        l2_leaf_reg=5, loss_function="Logloss", eval_metric="Logloss",
        random_seed=CFG["seed"], thread_count=2, allow_writing_files=False,
    )
    model.fit(
        X.loc[train_mask, columns], y.loc[train_mask],
        cat_features=[f for f in CATEGORICAL_FEATURES if f in columns],
        eval_set=(X.loc[stop_mask, columns], y.loc[stop_mask]),
        early_stopping_rounds=50, verbose=False,
    )
    models[name] = model
    model_columns[name] = columns
    training_rows.append({"model": name, "best_iteration": int(model.get_best_iteration())})
display(pd.DataFrame(training_rows))
print("Three classifiers fitted. The fixed rules baseline requires no fitting.")

## 7. Compare all detectors under the same review budgets

For each UTC day in the validation period, select up to K transactions with the highest score. Ties use transaction ID, which follows event order in this simulator. Every detector uses the same transactions and the same budgets.

This is an **offline end-of-day ranking diagnostic**: it looks across that day's scores. It is not a live blocking or queue simulation. A selected row is labelled REVIEW; an unselected row is NO_REVIEW, not an actual payment approval.

Fraud-value capture uses labelled attempted amounts, including failed attempts. It is not settled fraud loss or savings. Average precision is reported explicitly; it is not a trapezoidal PR-AUC calculation.

In [ ]:
def safe_ratio(numerator, denominator):
    return float(numerator / denominator) if denominator else np.nan

def select_daily(frame, k):
    ordered = frame.reset_index().sort_values(
        ["day", "score", "transaction_id"], ascending=[True, False, True])
    selected_ids = ordered.groupby("day", sort=False).head(k).transaction_id
    return pd.Series(frame.index.isin(selected_ids), index=frame.index)

def evaluate_scores(name, scores):
    frame = data.loc[val_mask, [
        "timestamp", "customer_id", "amount", "is_fraud", "fraud_scenario",
        "legitimate_context", "category", "country",
    ]].copy()
    frame["day"] = frame.timestamp.dt.floor("D")
    frame["score"] = np.asarray(scores)
    frame["model"] = name
    total_frauds = int(frame.is_fraud.sum())
    total_legit = int(frame.is_fraud.eq(0).sum())
    total_fraud_value = float(frame.loc[frame.is_fraud.eq(1), "amount"].sum())
    ap = float(average_precision_score(frame.is_fraud, frame.score))
    auc = float(roc_auc_score(frame.is_fraud, frame.score))
    rows, daily_rows = [], []
    for k in CFG["review_budgets"]:
        selected = select_daily(frame, k)
        picked = frame.loc[selected]
        hits = picked.loc[picked.is_fraud.eq(1)]
        legitimate_reviewed = int(picked.is_fraud.eq(0).sum())
        rows.append({
            "model": name, "daily_review_cap": k,
            "average_precision": ap, "roc_auc": auc,
            "review_count": len(picked), "fraud_reviewed": len(hits),
            "precision": safe_ratio(len(hits), len(picked)),
            "fraud_recall": safe_ratio(len(hits), total_frauds),
            "fraud_attempt_value_capture": safe_ratio(float(hits.amount.sum()), total_fraud_value),
            "legitimate_reviewed": legitimate_reviewed,
            "false_positive_rate": safe_ratio(legitimate_reviewed, total_legit),
        })
        for day, group in frame.groupby("day"):
            reviewed = group.loc[selected.loc[group.index]]
            day_hits = int(reviewed.is_fraud.sum())
            daily_rows.append({
                "model": name, "daily_review_cap": k, "day": str(day),
                "transactions": len(group), "frauds": int(group.is_fraud.sum()),
                "reviews": len(reviewed), "fraud_reviewed": day_hits,
                "precision": safe_ratio(day_hits, len(reviewed)),
                "recall": safe_ratio(day_hits, int(group.is_fraud.sum())),
            })
        if k == CFG["main_review_budget"]:
            frame["selected_for_review"] = selected
            frame["action"] = np.where(selected, "REVIEW", "NO_REVIEW")
    return rows, daily_rows, frame

score_map = {"rules": rule_scores(X.loc[val_mask])}
for name, model in models.items():
    score_map[name] = model.predict_proba(X.loc[val_mask, model_columns[name]])[:, 1]

all_metrics, all_daily, predictions = [], [], {}
for name, scores in score_map.items():
    metric_rows, daily_rows, frame = evaluate_scores(name, scores)
    all_metrics.extend(metric_rows)
    all_daily.extend(daily_rows)
    predictions[name] = frame
comparison = pd.DataFrame(all_metrics)
daily_metrics = pd.DataFrame(all_daily)
comparison.to_csv(OUT / "comparison.csv", index=False)
daily_metrics.to_csv(OUT / "daily_metrics.csv", index=False)
display(comparison.round(4))

# A provisional selection for investigation, made using validation only.
operating_point = comparison.loc[comparison.daily_review_cap.eq(CFG["main_review_budget"])]
best_name = operating_point.sort_values(
    ["precision", "fraud_attempt_value_capture", "average_precision", "model"],
    ascending=[False, False, False, True],
).iloc[0]["model"]
print("Provisional validation leader:", best_name)
print("This is development evidence, not a final test estimate.")

## 8. Examine scenarios and legitimate behaviour changes

Overall metrics can hide weak spots. We report recall separately for each injected fraud scenario and alert rates for legitimate contexts.

Context labels are mutually exclusive simulator annotations; for example, a large purchase while travelling is recorded as large_purchase in this first generator. They are used only for evaluation. Differences across these subsets are descriptive, not causal estimates of feature benefits.

In [ ]:
scenario_rows, context_rows = [], []
for name, frame in predictions.items():
    fraudulent = frame.loc[frame.is_fraud.eq(1)]
    for scenario, group in fraudulent.groupby("fraud_scenario"):
        selected = group.loc[group.selected_for_review]
        scenario_rows.append({
            "model": name, "scenario": scenario, "fraud_count": len(group),
            "fraud_reviewed": len(selected), "recall": safe_ratio(len(selected), len(group)),
            "attempt_value_capture": safe_ratio(float(selected.amount.sum()), float(group.amount.sum())),
        })
    legitimate = frame.loc[frame.is_fraud.eq(0)]
    for context, group in legitimate.groupby("legitimate_context"):
        context_rows.append({
            "model": name, "context": context, "legitimate_count": len(group),
            "legitimate_reviewed": int(group.selected_for_review.sum()),
            "alert_rate": float(group.selected_for_review.mean()),
        })
scenario_breakdown = pd.DataFrame(scenario_rows)
context_breakdown = pd.DataFrame(context_rows)
scenario_breakdown.to_csv(OUT / "scenario_breakdown.csv", index=False)
context_breakdown.to_csv(OUT / "legitimate_context_breakdown.csv", index=False)
display(scenario_breakdown.round(4))
display(context_breakdown.round(4))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for name, frame in predictions.items():
    precision, recall, _ = precision_recall_curve(frame.is_fraud, frame.score)
    axes[0].plot(recall, precision, label=name)
    subset = comparison.loc[comparison.model.eq(name)]
    axes[1].plot(subset.daily_review_cap, subset.fraud_recall, marker="o", label=name)
axes[0].set(xlabel="Recall", ylabel="Precision", title="Validation precision–recall")
axes[1].set(xlabel="Reviews available per day", ylabel="Fraud recall", title="Offline review-budget comparison")
for axis in axes:
    axis.legend(fontsize=8)
    axis.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(OUT / "validation_comparison.png", dpi=160)
plt.show()

## 9. Inspect an alert and export mistakes

Each model gets separate false-positive, missed-fraud and true-positive case files. Observed evidence and triggered rules are reported separately. These are factual feature summaries, not SHAP values or proof of the fraud mechanism.

The investigation view shows earlier attempts only. The displayed status column masks outcomes that were not yet available at the selected decision time. Ground-truth labels appear only in the explicitly retrospective evaluation row.

In [ ]:
def observed_evidence(row):
    facts = []
    if pd.notna(row.amount_ratio_30d):
        facts.append("amount is %.2fx the prior 30-day median attempted amount" % row.amount_ratio_30d)
    if row.new_device:
        facts.append("device not seen in this account's prior observations")
    if row.new_country:
        facts.append("country not seen in this account's prior observations")
    facts.append("%d earlier attempts in 1 hour" % row.attempts_1h)
    facts.append("%d failed outcomes known in 1 hour" % row.known_failures_1h)
    facts.append("%d previously observed accounts on the device" % row.prior_accounts_on_device)
    return "; ".join(facts)

val_features = X.loc[val_mask]
trigger_matrix = rule_triggers(val_features)
trigger_text = trigger_matrix.apply(
    lambda row: "; ".join(name for name in RULE_WEIGHTS if row[name]), axis=1)
evidence_text = val_features.apply(observed_evidence, axis=1)

for name, frame in predictions.items():
    frame["triggered_rules"] = trigger_text
    frame["observed_evidence"] = evidence_text
    false_positives = frame.loc[frame.selected_for_review & frame.is_fraud.eq(0)].nlargest(50, "score")
    missed_fraud = frame.loc[~frame.selected_for_review & frame.is_fraud.eq(1)].nlargest(50, "amount")
    true_positives = frame.loc[frame.selected_for_review & frame.is_fraud.eq(1)].nlargest(20, "score")
    cases = pd.concat([
        false_positives.assign(case_type="false_positive"),
        missed_fraud.assign(case_type="missed_fraud"),
        true_positives.assign(case_type="true_positive"),
    ])
    cases.to_csv(OUT / (name + "_error_cases.csv"), index_label="transaction_id")
    frame.to_csv(OUT / (name + "_validation_predictions.csv"), index_label="transaction_id")

def investigate(transaction_id, model_name=None):
    model_name = best_name if model_name is None else model_name
    frame = predictions[model_name]
    if transaction_id not in frame.index:
        raise KeyError("Choose a transaction ID from this model's validation predictions.")
    current = frame.loc[transaction_id]
    print("Retrospective evaluation — ground truth is visible below:")
    display(current.to_frame("value"))
    prior = transactions.loc[
        transactions.customer_id.eq(current.customer_id)
        & (transactions.timestamp < current.timestamp)
    ].tail(15).copy()
    prior["status_known_at_decision"] = np.where(
        prior.outcome_available_at <= current.timestamp, prior.status, "NOT_YET_AVAILABLE")
    display(prior[["transaction_id", "timestamp", "amount", "device_id", "country",
                   "category", "status_known_at_decision"]])
    print("Observed feature evidence; this is not a causal model explanation.")
    display(X.loc[[transaction_id]])

example_id = predictions[best_name].sort_values("score", ascending=False).index[0]
investigate(example_id)
# To investigate another case later:
# investigate("T00001234", model_name="catboost_history")

## 10. Save a reproducible run and download the results

The ZIP contains actual results from this execution: model comparisons, daily and scenario breakdowns, investigation cases, a chart, saved models, development-period transactions, and a manifest.

The manifest records the seed, dependency versions, data fingerprint, feature columns, rule weights, time boundaries, and timing checks. Saved models need the same feature builder and column order at inference; they are not standalone real-time services.

Next development step: compare the models at 50 reviews/day, read both false positives and missed fraud, propose a change, and test it on development data. Repeat the agreed experiments across seeds and temporal windows before making broad claims. Reserve final-test evaluation for the frozen candidate.

In [ ]:
for name, model in models.items():
    if name.startswith("catboost"):
        model.save_model(str(OUT / (name + ".cbm")))
    else:
        joblib.dump(model, OUT / (name + ".joblib"))
history_model = models["catboost_history"]
pd.DataFrame({
    "feature": FEATURES, "importance": history_model.get_feature_importance()
}).sort_values("importance", ascending=False).to_csv(OUT / "feature_importance.csv", index=False)
transactions.loc[transactions.timestamp < test_at].to_csv(
    OUT / "development_transactions.csv.gz", index=False, compression="gzip")
manifest = {
    "project": "transaction-fraud-intelligence",
    "implementation_version": "0.1.0",
    "run_id": RUN_ID, "configuration": CFG,
    "source": "self-generated synthetic payment attempts; no real customer data",
    "currency": "INR (synthetic standardized amounts)",
    "dataset_sha256": hashlib.sha256(
        pd.util.hash_pandas_object(transactions, index=False).values.tobytes()).hexdigest(),
    "features": FEATURES, "model_columns": model_columns,
    "categorical_features": CATEGORICAL_FEATURES,
    "rule_weights": RULE_WEIGHTS,
    "fit_at": str(fit_at), "validation_at": str(validation_at),
    "locked_test_at": str(test_at), "locked_test_evaluated": False,
    "timing_checks": timing_checks,
    "provisional_validation_leader": best_name,
    "model_parameters": {name: model.get_params() for name, model in models.items() if name.startswith("catboost")},
    "versions": {spec.split("==")[0]: importlib.metadata.version(spec.split("==")[0]) for spec in DEPENDENCIES},
    "limitations": [
        "Development-validation results under one simulator and one seed.",
        "Scenario labels and hidden customer profiles excluded from model inputs.",
        "Fraud mechanisms and legitimate lookalikes remain simplified.",
        "Seven-day confirmation for both classes is a simulation assumption.",
        "Daily top-K is offline transaction ranking, not online payment decisioning.",
        "Attempted fraudulent value includes failed attempts and is not loss or savings.",
        "Classifiers are uncalibrated; rules output points rather than probabilities.",
        "Evidence strings are observed features and rules, not causal explanations or SHAP.",
        "Final test has not been evaluated; dashboard and public-data benchmark are pending.",
    ],
}
(OUT / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
archive = Path(shutil.make_archive(str(OUT), "zip", root_dir=OUT))
print("Run complete. Download:", archive)
print("Start with comparison.csv, scenario_breakdown.csv, and the leader's error_cases.csv.")
print("The final test period remains unevaluated.")
if not SMOKE:
    try:
        from google.colab import files
    except ImportError:
        display(FileLink(str(archive)))
    else:
        files.download(str(archive))
else:
    display(FileLink(str(archive)))